# Medium-architecture five-seed run
Pre-specified intermediate-capacity ablation between the original/full and small architectures.

| Component | Full | **Medium (this notebook)** | Small |
|---|---:|---:|---:|
| Shared embedding | 64 | **48** | 32 |
| MRI channels | 16→32→64→128→256 | **12→24→48→96→192** | 8→16→32→64→128 |
| MRI transformer dimension | 256 | **192** | 128 |
| MRI transformer heads | 8 | **6** | 4 |
| MRI feed-forward dimension | 512 | **384** | 256 |
| MRI transformer layers | 1 | **1** | 1 |
| 3MT attention heads | 4 | **4** | 2 |
| Evidential hidden dimension | 32 | **24** | 16 |
| Trainable parameters | 4,428,114 | **2,507,438** | 1,130,218 |

The medium model retains **56.6% of the full model's trainable parameters** (a **43.4% reduction**), while keeping **all input features and modalities**.

Everything else is intentionally held fixed: the same five outer folds, seeds **17, 42, 73, 101, 137**, prepared inputs, loss, modality dropout, AdamW optimisation, scheduler, early stopping, and fixed **0.5 / 0.5 3MT–TMC fusion**.

**Interpretation note:** this architecture was proposed after inspecting earlier experimental results, so treat it as an exploratory capacity ablation rather than a new untouched-test model-selection stage. A final unbiased performance claim should rely on external validation (for example AIBL) or another genuinely untouched evaluation set.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import roc_auc_score, average_precision_score

PROJECT_ROOT = Path("/home/jovyan/thesis_data/adni_mri/adni_mri")
CODE_ROOT = Path("/home/jovyan/thesis_code")
OUTPUT_ROOT = Path("/home/jovyan/thesis_runs")

WORKER = CODE_ROOT / "ADNI_JHub_MediumArch_Worker.py"

SEEDS = [17, 42, 73, 101, 137]
FOLDS = [0, 1, 2, 3, 4]

MAX_EPOCHS = 50
BATCH_SIZE = 2

EXPERIMENT_NAME = "gated_cmt_fixed_equal_fusion_medium_v1_md050"
EXPERIMENT_ROOT = OUTPUT_ROOT / EXPERIMENT_NAME

EXPECTED_TRAINABLE_PARAMETERS = 2_507_438

print("Project root:", PROJECT_ROOT)
print("Worker:", WORKER)
print("Writable output root:", OUTPUT_ROOT)
print("Experiment:", EXPERIMENT_NAME)
print("Seeds:", SEEDS)
print("Folds:", FOLDS)
print("Total runs:", len(SEEDS) * len(FOLDS))
print("Expected trainable parameters:", f"{EXPECTED_TRAINABLE_PARAMETERS:,}")


In [ ]:
# Input check: fail before training if inputs, GPU, or the correct worker are missing.

assert PROJECT_ROOT.exists(), f"Missing project root: {PROJECT_ROOT}"
assert WORKER.exists(), (
    f"Missing worker script: {WORKER}\n"
    "Upload ADNI_JHub_MediumArch_Worker.py into /home/jovyan/thesis_code."
)

# Guard against accidentally uploading/running the full or small worker.
worker_text = WORKER.read_text(encoding="utf-8")
required_worker_tokens = [
    'ARCHITECTURE_PRESET = "medium_v1"',
    'EXPERIMENT_NAME = "gated_cmt_fixed_equal_fusion_medium_v1_md050"',
    'MODALITY_EMBEDDING_DIM = 48',
    'patch_embedding_dim=192',
    'transformer_heads=6',
    'transformer_feedforward_dim=384',
    'hidden_dim=24',
]
missing_tokens = [token for token in required_worker_tokens if token not in worker_text]
if missing_tokens:
    raise RuntimeError(
        "The worker does not match the intended medium architecture. "
        f"Missing markers: {missing_tokens}"
    )

input_root = (
    PROJECT_ROOT
    / "models"
    / "final_task_ready_inputs"
    / "mci_prognosis"
)

missing_folds = []
for fold in FOLDS:
    path = input_root / f"mci_prognosis_outer_fold_{fold}_final_task_ready.csv"
    if not path.exists():
        missing_folds.append(str(path))

if missing_folds:
    raise FileNotFoundError(
        "Missing prepared fold CSVs:\n" + "\n".join(missing_folds)
    )

print("All five prepared fold CSVs are present.")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
write_test = OUTPUT_ROOT / ".write_test"
write_test.write_text("ok", encoding="utf-8")
write_test.unlink()
print("Writable output directory is ready:", OUTPUT_ROOT)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory: {props.total_memory / (1024**3):.2f} GB")
else:
    raise RuntimeError(
        "CUDA is not available in this JHub kernel/session. "
        "Do not start the 25-run experiment on CPU."
    )

usage = os.statvfs(str(PROJECT_ROOT))
free_gb = usage.f_bavail * usage.f_frsize / (1024**3)
print(f"Free filesystem space: {free_gb:.1f} GB")

print("\nMedium architecture:")
print("  shared embedding: 48")
print("  MRI channels: 12 -> 24 -> 48 -> 96 -> 192")
print("  MRI transformer: dim=192, heads=6, FF=384, layers=1")
print("  3MT attention heads: 4")
print("  evidential hidden dim: 24")
print("  expected trainable parameters: 2,507,438")
print("  parameter reduction vs full: 43.4%")
print("\nInput check passed.")


## Run all five seeds × five folds

This launcher is restart-safe:

- completed seed/fold runs are skipped;
- an interrupted run with a saved last-epoch checkpoint is resumed;
- a run with no prior checkpoint starts fresh;
- the launcher stops immediately if one run fails.

Outputs are stored under:

`/home/jovyan/thesis_runs/gated_cmt_fixed_equal_fusion_medium_v1_md050/seed_<seed>/mci_prognosis/fold_<fold>/`


In [ ]:
for seed in SEEDS:
    for fold in FOLDS:
        fold_root = (
            EXPERIMENT_ROOT
            / f"seed_{seed}"
            / "mci_prognosis"
            / f"fold_{fold}"
        )

        test_metrics_path = (
            fold_root
            / "predictions"
            / "test_metrics.json"
        )

        last_checkpoint_path = (
            fold_root
            / "checkpoints"
            / "last_epoch_checkpoint.pt"
        )

        if test_metrics_path.exists():
            print("=" * 88)
            print(f"SKIP completed run | seed={seed} fold={fold}")
            print(test_metrics_path)
            continue

        mode = "resume" if last_checkpoint_path.exists() else "fresh"

        print("\n" + "=" * 88)
        print(f"START | seed={seed} fold={fold} mode={mode}")
        print("=" * 88)

        command = [
            sys.executable,
            "-u",
            str(WORKER),
            "--seed", str(seed),
            "--fold", str(fold),
            "--mode", mode,
            "--project-root", str(PROJECT_ROOT),
            "--output-root", str(OUTPUT_ROOT),
            "--max-epochs", str(MAX_EPOCHS),
            "--batch-size", str(BATCH_SIZE),
        ]

        result = subprocess.run(
            command,
            cwd=str(CODE_ROOT),
            env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )

        if result.returncode != 0:
            raise RuntimeError(
                f"Run failed for seed={seed}, fold={fold}. "
                "The launcher stopped so the problem can be fixed "
                "without starting later runs."
            )

print("\nAll requested seed/fold runs are complete.")


## Aggregate the 25 runs

This cell reads the worker prediction columns (`TARGET`, `FINAL_P_pMCI`) and accepts lowercase variants for compatibility. It reports:

- every seed × fold test result;
- pooled out-of-fold ROC AUC and average precision for each seed;
- five-seed mean ± SD;
- mean fold ROC AUC across all 25 runs;
- the recorded trainable-parameter count.

It also saves the summary CSV files under the medium experiment directory.


In [ ]:
def resolve_column(frame, candidates):
    for name in candidates:
        if name in frame.columns:
            return name
    raise KeyError(
        f"None of {candidates} are present. "
        f"Available columns: {frame.columns.tolist()}"
    )

fold_rows = []
seed_rows = []
recorded_parameter_counts = []

for seed in SEEDS:
    seed_prediction_tables = []

    for fold in FOLDS:
        fold_root = (
            EXPERIMENT_ROOT
            / f"seed_{seed}"
            / "mci_prognosis"
            / f"fold_{fold}"
        )

        metrics_path = fold_root / "predictions" / "test_metrics.json"
        predictions_path = fold_root / "predictions" / "test_predictions.csv"
        training_config_path = fold_root / "history" / "training_configuration.json"

        if not metrics_path.exists():
            fold_rows.append({
                "seed": seed,
                "fold": fold,
                "status": "missing",
            })
            continue

        if not predictions_path.exists():
            raise FileNotFoundError(
                f"Metrics exist but predictions are missing: {predictions_path}"
            )

        with metrics_path.open("r", encoding="utf-8") as f:
            metrics = json.load(f)

        final_metrics = metrics["test_metrics"]["final"]

        trainable_parameters = np.nan
        if training_config_path.exists():
            with training_config_path.open("r", encoding="utf-8") as f:
                training_config = json.load(f)
            trainable_parameters = int(training_config["trainable_parameters"])
            recorded_parameter_counts.append(trainable_parameters)

        fold_rows.append({
            "seed": seed,
            "fold": fold,
            "status": "complete",
            "trainable_parameters": trainable_parameters,
            "best_epoch": metrics["checkpoint_epoch"],
            "validation_roc_auc": metrics["checkpoint_validation_auc"],
            "test_roc_auc": final_metrics["roc_auc"],
            "test_average_precision": final_metrics["average_precision"],
            "test_balanced_accuracy": final_metrics["balanced_accuracy"],
            "test_brier_score": final_metrics["brier_score"],
            "test_ece": final_metrics["expected_calibration_error"],
            "test_mean_uncertainty": final_metrics["mean_uncertainty"],
            "test_participants": metrics["test_participants"],
        })

        pred = pd.read_csv(predictions_path)
        pred["seed"] = seed
        pred["fold"] = fold
        seed_prediction_tables.append(pred)

    if len(seed_prediction_tables) == len(FOLDS):
        pooled = pd.concat(seed_prediction_tables, ignore_index=True)

        target_col = resolve_column(pooled, ["TARGET", "target"])
        hybrid_prob_col = resolve_column(
            pooled,
            ["FINAL_P_pMCI", "final_p_pMCI"],
        )

        seed_fold_aucs = [
            row["test_roc_auc"]
            for row in fold_rows
            if row.get("seed") == seed
            and row.get("status") == "complete"
        ]

        seed_rows.append({
            "seed": seed,
            "pooled_test_participants": len(pooled),
            "pooled_hybrid_roc_auc": roc_auc_score(
                pooled[target_col],
                pooled[hybrid_prob_col],
            ),
            "pooled_hybrid_average_precision": average_precision_score(
                pooled[target_col],
                pooled[hybrid_prob_col],
            ),
            "mean_fold_roc_auc": float(np.mean(seed_fold_aucs)),
        })

fold_summary = pd.DataFrame(fold_rows)
seed_summary = pd.DataFrame(seed_rows)

print("Fold-level results:")
display(fold_summary)

print("\nSeed-level pooled out-of-fold results:")
display(seed_summary)

if recorded_parameter_counts:
    unique_parameter_counts = sorted(set(recorded_parameter_counts))
    print("\nRecorded trainable-parameter counts:", unique_parameter_counts)
    if unique_parameter_counts != [EXPECTED_TRAINABLE_PARAMETERS]:
        raise RuntimeError(
            "Unexpected model parameter count. "
            f"Expected {EXPECTED_TRAINABLE_PARAMETERS:,}, "
            f"recorded {unique_parameter_counts}."
        )

if len(seed_summary) == len(SEEDS):
    mean_auc = seed_summary["pooled_hybrid_roc_auc"].mean()
    sd_auc = seed_summary["pooled_hybrid_roc_auc"].std(ddof=1)

    mean_ap = seed_summary["pooled_hybrid_average_precision"].mean()
    sd_ap = seed_summary["pooled_hybrid_average_precision"].std(ddof=1)

    complete_fold_rows = fold_summary.loc[
        fold_summary["status"] == "complete"
    ].copy()

    mean_all_25_fold_auc = complete_fold_rows["test_roc_auc"].mean()
    sd_all_25_fold_auc = complete_fold_rows["test_roc_auc"].std(ddof=1)

    print("\nFive-seed pooled hybrid ROC AUC:")
    print(f"{mean_auc:.6f} ± {sd_auc:.6f}")

    print("\nFive-seed pooled hybrid average precision:")
    print(f"{mean_ap:.6f} ± {sd_ap:.6f}")

    print("\nMean fold ROC AUC across all 25 runs:")
    print(f"{mean_all_25_fold_auc:.6f} ± {sd_all_25_fold_auc:.6f}")

    summary_dir = EXPERIMENT_ROOT / "five_seed_summary"
    summary_dir.mkdir(parents=True, exist_ok=True)

    fold_summary.to_csv(
        summary_dir / "fold_metrics_all_seeds.csv",
        index=False,
    )

    seed_summary.to_csv(
        summary_dir / "pooled_metrics_by_seed.csv",
        index=False,
    )

    print("\nSaved summary files to:")
    print(summary_dir)

else:
    print(
        f"\nWARNING: pooled results are available for "
        f"{len(seed_summary)}/{len(SEEDS)} seeds."
    )


## Optional descriptive capacity comparison

After the medium run finishes, this cell looks for the existing small- and full-architecture summary files and puts the available results side by side.

This is **descriptive**, not a fresh unbiased model-selection test, because the medium architecture was proposed after earlier results had already been inspected.


In [ ]:
architecture_paths = {
    "small_1.13M": (
        OUTPUT_ROOT
        / "gated_cmt_fixed_equal_fusion_small_v1_md050"
        / "five_seed_summary"
        / "pooled_metrics_by_seed.csv"
    ),
    "medium_2.51M": (
        EXPERIMENT_ROOT
        / "five_seed_summary"
        / "pooled_metrics_by_seed.csv"
    ),
    "full_4.43M": (
        OUTPUT_ROOT
        / "gated_cmt_fixed_equal_fusion_full_arch_md050"
        / "five_seed_summary"
        / "pooled_metrics_by_seed.csv"
    ),
}

comparison_rows = []

for architecture, path in architecture_paths.items():
    if not path.exists():
        print(f"Not available yet: {architecture} -> {path}")
        continue

    df = pd.read_csv(path)

    comparison_rows.append({
        "architecture": architecture,
        "n_seeds": len(df),
        "pooled_roc_auc_mean": df["pooled_hybrid_roc_auc"].mean(),
        "pooled_roc_auc_sd": df["pooled_hybrid_roc_auc"].std(ddof=1),
        "pooled_ap_mean": df["pooled_hybrid_average_precision"].mean(),
        "pooled_ap_sd": df["pooled_hybrid_average_precision"].std(ddof=1),
        "mean_fold_roc_auc_across_seeds": df["mean_fold_roc_auc"].mean(),
    })

capacity_comparison = pd.DataFrame(comparison_rows)

if len(capacity_comparison):
    display(
        capacity_comparison.sort_values(
            "pooled_roc_auc_mean",
            ascending=False,
        ).reset_index(drop=True)
    )

    comparison_out = (
        EXPERIMENT_ROOT
        / "five_seed_summary"
        / "capacity_comparison_descriptive.csv"
    )
    comparison_out.parent.mkdir(parents=True, exist_ok=True)
    capacity_comparison.to_csv(comparison_out, index=False)
    print("Saved:", comparison_out)
else:
    print("No architecture summary files are available yet.")
